# 02 - Baseline TF-IDF Retriever

This notebook builds the baseline model for the cardiovascular chatbot.

The baseline uses:
- TF-IDF vectorization
- Cosine similarity
- Top-k document chunk retrieval
- Extractive response generation from retrieved chunks

This baseline does not use transfer learning or large language models.

In [1]:
import json
import re
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
BASE_DIR = Path.cwd()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = BASE_DIR / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)
print("Results directory:", RESULTS_DIR)

Base directory: d:\CardioBot_NLP_Final
Raw directory: d:\CardioBot_NLP_Final\data\raw
Processed directory: d:\CardioBot_NLP_Final\data\processed
Results directory: d:\CardioBot_NLP_Final\results


In [3]:
def read_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data


test_path = PROCESSED_DIR / "test.jsonl"
test_data = read_jsonl(test_path)

print("Test data size:", len(test_data))
pd.DataFrame(test_data).head()

Test data size: 34


,id,topic,source,question,answer
0,qa_047,Cardiomyopathy,Cardiomyopathy.txt,What is dilated cardiomyopathy?,Dilated cardiomyopathy is a type of cardiomyop...
1,qa_057,Heart Valve Disease,Heart Valve Disease.txt,What is valve regurgitation?,Valve regurgitation happens when valve flaps d...
2,qa_099,Blood Flow,Blood_Flow.txt,What are the main functions of blood flow?,Blood flow delivers oxygen and nutrients to or...
3,qa_135,Heart Failure,heart_failure.txt,What are the ACC/AHA stages of heart failure?,The ACC/AHA stages of heart failure are Stage ...
4,qa_070,Stroke,Stroke.txt,How is hemorrhagic stroke treated?,Hemorrhagic stroke treatment focuses on contro...


In [4]:
raw_files = list(RAW_DIR.glob("*.txt"))

documents = []

for file_path in raw_files:
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    
    documents.append({
        "source": file_path.name,
        "text": text
    })

print("Total raw documents:", len(documents))

for doc in documents[:5]:
    print("-", doc["source"])

Total raw documents: 29
- Angiography.txt
- Angioplasty and stent.txt
- Arrhythmia.txt
- Atherosclerosis.txt
- Blood pressure measurement.txt


In [5]:
def clean_text(text):
    text = text.replace("\r", "\n")
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


for doc in documents:
    doc["text"] = clean_text(doc["text"])

print("Cleaning completed.")
print(documents[0]["text"][:500])

Cleaning completed.
Angiography
Definition
Angiography is a type of X-ray imaging used to examine blood vessels and assess blood flow. Because blood vessels do not appear clearly on a normal X-ray, a special dye called a contrast agent is injected into the bloodstream to make them visible. This allows doctors to detect abnormalities or blockages. The images produced during this procedure are called angiograms.
Purpose
Angiography is used to evaluate the condition of blood vessels and how blood flows through them. I


In [6]:
def chunk_text(documents, chunk_size=180, overlap=30):
    chunks = []

    for doc in documents:
        text = doc["text"]
        paragraphs = [p.strip() for p in text.split("\n") if p.strip()]

        for para in paragraphs:
            if len(para.split()) < 20:
                continue

            sentences = re.split(r'(?<=[.!?])\s+', para)

            current_chunk = []
            current_length = 0

            for sentence in sentences:
                words = sentence.split()

                if current_length + len(words) <= chunk_size:
                    current_chunk.append(sentence)
                    current_length += len(words)
                else:
                    chunk_str = " ".join(current_chunk).strip()

                    if len(chunk_str.split()) >= 30:
                        chunks.append({
                            "source": doc["source"],
                            "text": chunk_str
                        })

                    current_chunk = current_chunk[-1:]
                    current_length = sum(len(s.split()) for s in current_chunk)

                    current_chunk.append(sentence)
                    current_length += len(words)

            if current_chunk:
                chunk_str = " ".join(current_chunk).strip()

                if len(chunk_str.split()) >= 30:
                    chunks.append({
                        "source": doc["source"],
                        "text": chunk_str
                    })

    # remove duplicate chunks
    unique_chunks = []
    seen = set()

    for chunk in chunks:
        if chunk["text"] not in seen:
            unique_chunks.append(chunk)
            seen.add(chunk["text"])

    return unique_chunks


chunks = chunk_text(documents)

print("Total chunks:", len(chunks))
pd.DataFrame(chunks).head()

Total chunks: 303


,source,text
0,Angiography.txt,Angiography is a type of X-ray imaging used to...
1,Angiography.txt,Angiography is used to evaluate the condition ...
2,Angiography.txt,Angiography is usually performed in a hospital...
3,Angiography.txt,Angiography is generally considered safe and p...
4,Angiography.txt,There are several types of angiography dependi...


In [8]:
chunk_df = pd.DataFrame(chunks)

chunk_distribution = chunk_df["source"].value_counts().reset_index()
chunk_distribution.columns = ["source", "chunk_count"]

chunk_distribution

,source,chunk_count
0,Echocardiogram.txt,26
1,Pacemaker.txt,21
2,Cholesterol.txt,16
3,Stress test.txt,16
4,Cardiac ablation.txt,15
5,Arrhythmia.txt,14
6,Electrocardiogram.txt,14
7,Stroke.txt,14
8,heart_anatomy_and_function.txt,13
9,heart_disease_risk_factors.txt,12


# Build tf-idf

In [9]:
texts = [chunk["text"] for chunk in chunks]

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=10000
)

tfidf_matrix = vectorizer.fit_transform(texts)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (303, 7613)


In [10]:
# retriever
def retrieve_tfidf(question, top_k=3):
    query_vec = vectorizer.transform([question])
    scores = cosine_similarity(query_vec, tfidf_matrix)[0]

    top_indices = scores.argsort()[-top_k:][::-1]

    results = []
    for idx in top_indices:
        results.append({
            "source": chunks[idx]["source"],
            "text": chunks[idx]["text"],
            "score": float(scores[idx])
        })

    return results

In [11]:
# test retriever manual
sample_question = "What is a pacemaker?"

retrieved = retrieve_tfidf(sample_question, top_k=3)

print("Question:", sample_question)
print()

for i, item in enumerate(retrieved, start=1):
    print(f"Rank {i}")
    print("Source:", item["source"])
    print("Score:", round(item["score"], 4))
    print("Text:", item["text"][:500])
    print("-" * 80)

Question: What is a pacemaker?

Rank 1
Source: Pacemaker.txt
Score: 0.2576
Text: Security systems. Passing through an airport metal detector won't interfere with a pacemaker. But the metal in the pacemaker could sound the alarm. Do not stay too long near a metal-detection system. Carry an ID card that says you have a pacemaker.
--------------------------------------------------------------------------------
Rank 2
Source: Pacemaker.txt
Score: 0.2286
Text: A pacemaker is a small, battery-powered device that prevents the heart from beating too slowly. You need surgery to get a pacemaker. The device is placed under the skin near the collarbone.
--------------------------------------------------------------------------------
Rank 3
Source: Pacemaker.txt
Score: 0.226
Text: Usually, medicine is used to numb the skin where the pacemaker will be inserted. This medicine is called local anesthesia. During the pacemaker surgery, you may be fully awake or lightly sedated.
-------------------------

In [16]:
# baseline response
def generate_baseline_response(question, top_k=3, max_sentences=5, min_score=0.05):
    retrieved = retrieve_tfidf(question, top_k=top_k)

    best_score = retrieved[0]["score"]

    if best_score < min_score:
        return {
            "answer": "The information is not available in the cardiovascular knowledge base.",
            "retrieved": retrieved
        }

    combined_text = " ".join([item["text"] for item in retrieved])

    sentences = re.split(r'(?<=[.!?])\s+', combined_text)

    unique_sentences = []
    seen = set()

    for sentence in sentences:
        sentence = sentence.strip()
        if sentence and sentence not in seen:
            unique_sentences.append(sentence)
            seen.add(sentence)

    answer = " ".join(unique_sentences[:max_sentences])

    return {
        "answer": answer,
        "retrieved": retrieved
    }

In [17]:
sample_question = "What are the symptoms of stroke?"

result = generate_baseline_response(sample_question)

print("Question:")
print(sample_question)
print("\nBaseline Answer:")
print(result["answer"])

print("\nRetrieved Sources:")
for item in result["retrieved"]:
    print("-", item["source"], "| score:", round(item["score"], 4))

Question:
What are the symptoms of stroke?

Baseline Answer:
Call 911 (or your local emergency services number) if you think you’re experiencing stroke symptoms again. Another stroke has an even higher risk of causing severe complications and being fatal. Don’t wait to call for help or go to the emergency room. Stroke rehab is an important part of stroke treatment. You’ll need rehab to help you adjust to changes in your brain and body after a stroke.

Retrieved Sources:
- Stroke.txt | score: 0.2608
- Stroke.txt | score: 0.1254
- Stroke.txt | score: 0.1039


In [18]:
# run baseline ke test data
baseline_results = []

for i, item in enumerate(test_data, start=1):
    question = item["question"]
    reference = item["answer"]

    result = generate_baseline_response(question, top_k=3)

    baseline_results.append({
        "id": item["id"],
        "topic": item["topic"],
        "source": item["source"],
        "question": question,
        "reference_answer": reference,
        "baseline_answer": result["answer"],
        "top_source": result["retrieved"][0]["source"],
        "top_score": result["retrieved"][0]["score"],
        "retrieved_contexts": result["retrieved"]
    })

    print(f"[{i}/{len(test_data)}] {question}")

[1/34] What is dilated cardiomyopathy?
[2/34] What is valve regurgitation?
[3/34] What are the main functions of blood flow?
[4/34] What are the ACC/AHA stages of heart failure?
[5/34] How is hemorrhagic stroke treated?
[6/34] What is the heart?
[7/34] What are common symptoms of cardiomyopathy?
[8/34] How can acquired cardiomyopathy risk be reduced?
[9/34] What should someone do if heart attack warning signs occur?
[10/34] Where does cholesterol come from?
[11/34] Why are high blood pressure, high cholesterol, and smoking considered major heart disease risk factors?
[12/34] What is valve prolapse?
[13/34] What are the benefits of cardiac ablation?
[14/34] What is a Holter monitor?
[15/34] What are the main types of heart disease?
[16/34] When might an ECG be needed?
[17/34] What is systemic circulation?
[18/34] What is the correct way to measure blood pressure?
[19/34] What are risk factors for stroke?
[20/34] Why is managing diabetes important for heart disease prevention?
[21/34] Wh

In [19]:
baseline_df = pd.DataFrame(baseline_results)

baseline_df[[
    "id", 
    "topic", 
    "question", 
    "top_source", 
    "top_score", 
    "baseline_answer"
]].head()

,id,topic,question,top_source,top_score,baseline_answer
0,qa_047,Cardiomyopathy,What is dilated cardiomyopathy?,Cardiomyopathy.txt,0.292192,Dilated cardiomyopathy. In this type of cardio...
1,qa_057,Heart Valve Disease,What is valve regurgitation?,Echocardiogram.txt,0.368980,Heart valve disease. An echocardiogram can sho...
2,qa_099,Blood Flow,What are the main functions of blood flow?,Blood_Flow.txt,0.303583,"Blood flow has two main functions, which are d..."
3,qa_135,Heart Failure,What are the ACC/AHA stages of heart failure?,heart_failure.txt,0.358101,"Heart failure, also known as congestive heart ..."
4,qa_070,Stroke,How is hemorrhagic stroke treated?,Stroke.txt,0.227066,"If you have a hemorrhagic stroke, your provide..."


In [20]:
# save baseline result
baseline_output_path = RESULTS_DIR / "baseline_tfidf_answers.json"

with open(baseline_output_path, "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, indent=2, ensure_ascii=False)

print("Saved baseline results to:")
print(baseline_output_path)

Saved baseline results to:
d:\CardioBot_NLP_Final\results\baseline_tfidf_answers.json


In [23]:
# evaluasi source match
baseline_df["source_match"] = baseline_df["source"] == baseline_df["top_source"]

source_match_accuracy = baseline_df["source_match"].mean()

print("Source Match Accuracy:", round(source_match_accuracy, 4))
print("Correct:", baseline_df["source_match"].sum())
print("Total:", len(baseline_df))

Source Match Accuracy: 0.6176
Correct: 21
Total: 34


In [24]:
# evaluasi top-k score match
def check_topk_source_match(row):
    expected_source = row["source"]
    retrieved_sources = [item["source"] for item in row["retrieved_contexts"]]
    return expected_source in retrieved_sources


baseline_df["top3_source_match"] = baseline_df.apply(check_topk_source_match, axis=1)

top3_source_accuracy = baseline_df["top3_source_match"].mean()

print("Top-3 Source Match Accuracy:", round(top3_source_accuracy, 4))
print("Correct:", baseline_df["top3_source_match"].sum())
print("Total:", len(baseline_df))

Top-3 Source Match Accuracy: 0.8529
Correct: 29
Total: 34


In [25]:
def compute_text_similarity(text1, text2):
    pair_matrix = vectorizer.transform([text1, text2])
    sim = cosine_similarity(pair_matrix[0], pair_matrix[1])[0][0]
    return float(sim)


baseline_df["answer_similarity"] = baseline_df.apply(
    lambda row: compute_text_similarity(row["reference_answer"], row["baseline_answer"]),
    axis=1
)

print("Average Answer Similarity:", round(baseline_df["answer_similarity"].mean(), 4))

baseline_df[["question", "answer_similarity"]].head()

Average Answer Similarity: 0.2598


,question,answer_similarity
0,What is dilated cardiomyopathy?,0.380215
1,What is valve regurgitation?,0.347301
2,What are the main functions of blood flow?,0.360067
3,What are the ACC/AHA stages of heart failure?,0.130462
4,How is hemorrhagic stroke treated?,0.486735


In [26]:
baseline_summary = {
    "model": "TF-IDF Baseline",
    "test_size": len(baseline_df),
    "source_match_accuracy": round(float(source_match_accuracy), 4),
    "top3_source_match_accuracy": round(float(top3_source_accuracy), 4),
    "average_answer_similarity": round(float(baseline_df["answer_similarity"].mean()), 4),
    "average_top_score": round(float(baseline_df["top_score"].mean()), 4)
}

baseline_summary

{'model': 'TF-IDF Baseline',
 'test_size': 34,
 'source_match_accuracy': 0.6176,
 'top3_source_match_accuracy': 0.8529,
 'average_answer_similarity': 0.2598,
 'average_top_score': 0.2919}

In [27]:
summary_path = RESULTS_DIR / "baseline_tfidf_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(baseline_summary, f, indent=2, ensure_ascii=False)

baseline_df.to_csv(RESULTS_DIR / "baseline_tfidf_results.csv", index=False)

print("Saved baseline summary to:", summary_path)
print("Saved baseline CSV to:", RESULTS_DIR / "baseline_tfidf_results.csv")

Saved baseline summary to: d:\CardioBot_NLP_Final\results\baseline_tfidf_summary.json
Saved baseline CSV to: d:\CardioBot_NLP_Final\results\baseline_tfidf_results.csv


In [28]:
display_columns = [
    "question",
    "source",
    "top_source",
    "source_match",
    "top_score",
    "answer_similarity"
]

baseline_df[display_columns].sort_values("answer_similarity", ascending=True).head(10)

,question,source,top_source,source_match,top_score,answer_similarity
5,What is the heart?,heart_anatomy_and_function.txt,Electrocardiogram.txt,False,0.288698,0.027311
21,What are the main types of pacemakers?,Pacemaker.txt,Cardiomyopathy.txt,False,0.199643,0.050880
24,What is atherosclerosis?,Atherosclerosis.txt,Atherosclerosis.txt,True,0.131682,0.066883
15,When might an ECG be needed?,Electrocardiogram.txt,Holter Monitor.txt,False,0.167260,0.069097
20,What are possible risks of angiography?,Angiography.txt,Angiography.txt,True,0.235595,0.085664
9,Where does cholesterol come from?,Cholesterol.txt,Cholesterol.txt,True,0.186779,0.096281
25,What lifestyle changes can help coronary heart...,treatment_for_heart_disease.txt,Heart_Disease.txt,False,0.248391,0.107366
3,What are the ACC/AHA stages of heart failure?,heart_failure.txt,heart_failure.txt,True,0.358101,0.130462
28,What complications can untreated arrhythmia ca...,Arrhythmia.txt,Coronary Artery Disease.txt,False,0.132156,0.133726
7,How can acquired cardiomyopathy risk be reduced?,Cardiomyopathy.txt,Cardiomyopathy.txt,True,0.352538,0.146798
